In [ ]:
import sys
import os
import torch
import torch.nn as nn
import gc
import pandas as pd
import numpy as np
# from google.colab import drive

# drive.mount('/content/drive')
# PROJECT_PATH = '/content/drive/MyDrive/patch-tst'
# if PROJECT_PATH not in sys.path:
#     sys.path.append(PROJECT_PATH)
PROJECT_PATH = "." 

from src.model.patch_tst import PatchTST
from src.data_loader import get_dataloader
from config import config

target_file = os.path.join(PROJECT_PATH, "data/weather.csv")
dataset_name = os.path.basename(target_file).split('.')[0]

L_windows = [24, 48, 96, 192, 336, 720]
P_windows = [96, 192, 336, 720]
results_table = []

def evaluate_metrics(model, file_path, lookback, pred_len):
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    test_loader, _ = get_dataloader(file_path, config["batch_size"], flag='test', size=(lookback, pred_len))

    mse_fn, mae_fn = nn.MSELoss(), nn.L1Loss()
    mses, maes = [] , []

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            output = model(batch_x)
            mses.append(mse_fn(output, batch_y).item())
            maes.append(mae_fn(output, batch_y).item())

    return np.mean(mses), np.mean(maes)

def train_with_transfer(file_path, lookback, pred_len, pretrain_model_path=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader, train_set = get_dataloader(
        file_path, config["batch_size"], flag='train',
        size=(lookback, pred_len)
    )
    n_channels = train_set.data_x.shape[1]

    model = PatchTST(
        n_channels=n_channels,
        lookback_len=lookback,
        patch_len=config["patch_len"],
        stride=config["stride"],
        d_model=config["d_model"],
        n_heads=config["n_heads"],
        d_ff=config["d_ff"],
        n_layers=config["n_layers"],
        pred_len=pred_len
    ).to(device)

    if pretrain_model_path and os.path.exists(pretrain_model_path):
        print(f"Loading weights from: {pretrain_model_path}")
        pretrained_dict = torch.load(pretrain_model_path, map_location=device)
        model_dict = model.state_dict()

        transfer_dict = {
            k: v for k, v in pretrained_dict.items()
            if k in model_dict and v.size() == model_dict[k].size()
        }

        model_dict.update(transfer_dict)
        model.load_state_dict(model_dict)
        print(f"Successfully transferred {len(transfer_dict)} layers.")
    else:
        print("Pretrained model not found or path incorrect. Training from scratch.")

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    criterion = nn.MSELoss()

    model.train()
    for epoch in range(config["epochs"]):
        epoch_loss = 0
        for i, (batch_x, batch_y) in enumerate(train_loader):
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

    save_name = f"patch_tst_{dataset_name}_L{lookback}_P{pred_len}_transfer.pth"
    torch.save(model.state_dict(), os.path.join(PROJECT_PATH, save_name))

    return model

for P in P_windows:
    for L in L_windows:
        print(f"\n>>> TRANSFER LEARNING: L={L}, P={P} <<<")
        current_pretrain = os.path.join(PROJECT_PATH, f"patch_tst_electricity_L{L}_P{P}.pth")
        model = train_with_transfer(target_file, L, P, pretrain_model_path=current_pretrain)
        mse, mae = evaluate_metrics(model, target_file, L, P)
        results_table.append({
            'Lookback': L,
            'Predict': P,
            'MSE': mse,
            'MAE': mae
        })

        print(f"FINISHED L={L}, P={P} | MSE: {mse:.4f}, MAE: {mae:.4f}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

df_transfer = pd.DataFrame(results_table)
df_transfer.to_csv(os.path.join(PROJECT_PATH, f"results_transfer_{dataset_name}.csv"), index=False)